***INFERENCE AND SUBMISSION***

In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from sklearn.preprocessing import LabelEncoder
from scipy.ndimage import gaussian_filter1d

device = torch.device('cpu')
base_path = '/kaggle/input/competitions/birdclef-2026'
model_weights_path = '/kaggle/input/datasets/adrianosemerano/birdclef-weights/best_phaseB_Asymmetric_PCEN_Mixup.pth'

model_type = 'Asymmetric'
use_pcen = True
use_se = False

train_csv = os.path.join(base_path, 'train.csv')
df = pd.read_csv(train_csv)
label_encoder = LabelEncoder()
label_encoder.fit(df['primary_label'])
num_classes = len(label_encoder.classes_)

class AudioToSpectrogramGPU(nn.Module):
    def __init__(self, sr=32000, n_mels=256, n_fft=2048, hop_length=512, f_min=40, f_max=15000, use_pcen=True, use_specaugment=True):
        super().__init__()
        self.use_pcen = use_pcen
        self.use_specaugment = use_specaugment
        
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr, n_fft=n_fft, hop_length=hop_length,
            f_min=f_min, f_max=f_max, n_mels=n_mels, power=2.0 
        )
        self.eps = 1e-6
        self.s = 0.025
        self.alpha = 0.98
        self.delta = 2.0
        self.r = 0.5
        self.amp_to_db = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=80)
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=36)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=64)

    def forward(self, waveform):
        x = self.mel_spec(waveform)
        if self.use_pcen:
            ema = x.clone()
            for t in range(1, x.size(-1)):
                ema[..., t] = (1 - self.s) * ema[..., t - 1] + self.s * x[..., t]
            x = (x / (self.eps + ema)**self.alpha + self.delta)**self.r - self.delta**self.r
        else:
            x = self.amp_to_db(x)
            
        x = (x - x.mean()) / (x.std() + 1e-6)
        
        if self.training and self.use_specaugment:
            x = self.freq_mask(x)
            x = self.time_mask(x)
        return x

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, pool_freq=2, pool_time=1, model_type='Asymmetric', use_se=True):
        super().__init__()
        self.use_se = use_se
        
        if model_type == 'Asymmetric':
            self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=(5, 1), padding=(2, 0), bias=False)
            self.bn1 = nn.BatchNorm2d(out_channels)
            self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=(1, 5), padding=(0, 2), bias=False)
            self.bn2 = nn.BatchNorm2d(out_channels)
        else:
            self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
            self.bn1 = nn.BatchNorm2d(out_channels)
            self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
            self.bn2 = nn.BatchNorm2d(out_channels)
            
        if self.use_se:
            self.se = SEBlock(out_channels)
            
        self.pool = nn.MaxPool2d(kernel_size=(pool_freq, pool_time))

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        if self.use_se:
            x = self.se(x)
        x = self.pool(x)
        return x

class ResNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, use_se=True):
        super().__init__()
        self.use_se = use_se
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        if self.use_se:
            self.se = SEBlock(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.use_se:
            out = self.se(out) 
        out += self.shortcut(x)
        return F.relu(out)

class AttentivePooling(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.attention = nn.Conv1d(in_channels, num_classes, kernel_size=1, bias=True)
        self.classifier = nn.Conv1d(in_channels, num_classes, kernel_size=1, bias=True)

    def forward(self, x):
        att_weights = torch.softmax(self.attention(x), dim=-1)
        frame_logits = self.classifier(x)
        clip_logits = torch.sum(att_weights * frame_logits, dim=-1)
        return clip_logits, frame_logits

class BirdSED_Model(nn.Module):
    def __init__(self, num_classes, model_type='Asymmetric', use_pcen=True, use_se=True, use_specaugment=True):
        super().__init__()
        self.model_type = model_type
        self.audio_extractor = AudioToSpectrogramGPU(use_pcen=use_pcen, use_specaugment=use_specaugment)
        
        if self.model_type in ['Asymmetric', 'Standard']:
            self.stem = nn.Sequential(
                nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2, bias=False),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
            )
            self.block1 = CNNBlock(32, 64, pool_freq=2, pool_time=1, model_type=model_type, use_se=use_se)
            self.block2 = CNNBlock(64, 128, pool_freq=2, pool_time=2, model_type=model_type, use_se=use_se)
            self.block3 = CNNBlock(128, 256, pool_freq=2, pool_time=2, model_type=model_type, use_se=use_se)
            self.block4 = CNNBlock(256, 512, pool_freq=2, pool_time=2, model_type=model_type, use_se=use_se)
            
        elif self.model_type == 'ResNet':
            self.conv1 = nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2, bias=False)
            self.bn1 = nn.BatchNorm2d(32)
            self.pool1 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
            self.in_channels = 32
            self.layer1 = self._make_resnet_layer(64, stride=1, use_se=use_se)
            self.layer2 = self._make_resnet_layer(128, stride=2, use_se=use_se)
            self.layer3 = self._make_resnet_layer(256, stride=2, use_se=use_se)
            self.layer4 = self._make_resnet_layer(512, stride=2, use_se=use_se)
            
        self.freq_pool = nn.AdaptiveAvgPool2d((1, None))
        self.dropout = nn.Dropout(0.5)
        self.sed_head = AttentivePooling(in_channels=512, num_classes=num_classes)

    def _make_resnet_layer(self, out_channels, stride, use_se):
        layer = ResNetBlock(self.in_channels, out_channels, stride, use_se=use_se)
        self.in_channels = out_channels
        return layer

    def forward(self, waveform):
        x = self.audio_extractor(waveform)
        
        if self.model_type in ['Asymmetric', 'Standard']:
            x = self.stem(x)
            x = self.block4(self.block3(self.block2(self.block1(x))))
        elif self.model_type == 'ResNet':
            x = F.relu(self.bn1(self.conv1(x)))
            x = self.pool1(x)
            x = self.layer1(x)
            x = self.layer2(x)
            x = self.layer3(x)
            x = self.layer4(x)
            
        x = self.freq_pool(x).squeeze(2)
        x = self.dropout(x)
        clip_logits, _ = self.sed_head(x)
        return clip_logits

def get_padded_chunk(wf, start, end, chunk_samples):
    c_len = end - start
    pad_left = max(0, -start)
    start_safe = max(0, start)
    end_safe = min(end, wf.shape[1])
    pad_right = max(0, end - wf.shape[1])
    
    chunk = wf[:, start_safe:end_safe]
    if pad_left > 0 or pad_right > 0:
        chunk = torch.nn.functional.pad(chunk, (pad_left, pad_right))
    return chunk

def apply_temporal_smoothing(probs_matrix, sigma=1.0):
    smoothed_probs = np.zeros_like(probs_matrix)
    for class_idx in range(probs_matrix.shape[1]):
        smoothed_probs[:, class_idx] = gaussian_filter1d(probs_matrix[:, class_idx], sigma=sigma)
    return smoothed_probs

model = BirdSED_Model(num_classes=num_classes, model_type=model_type, use_pcen=use_pcen, use_se=use_se, use_specaugment=False)
model.load_state_dict(torch.load(model_weights_path, map_location=device))
model.to(device)
model.eval()

sample_sub_path = os.path.join(base_path, 'sample_submission.csv')
sample_df = pd.read_csv(sample_sub_path)
expected_species_columns = sample_df.columns[1:] 
test_audio_dir = os.path.join(base_path, 'test_soundscapes')

predictions_dict = {}

predicted_species_names = label_encoder.inverse_transform(range(num_classes))
species_to_idx = {sp: idx for idx, sp in enumerate(predicted_species_names)}

if os.path.exists(test_audio_dir) and len([f for f in os.listdir(test_audio_dir) if f.endswith('.ogg')]) > 0:
    test_files = [f for f in os.listdir(test_audio_dir) if f.endswith('.ogg')]
    print(f"Found {len(test_files)} hidden test files. Processing dynamically on CPU with TTA...")
    
    for file in test_files:
        file_path = os.path.join(test_audio_dir, file)
        file_stem = file.replace('.ogg', '')
        
        try:
            waveform, sr = torchaudio.load(file_path)
            if waveform.shape[0] > 1: 
                waveform = torch.mean(waveform, dim=0, keepdim=True)
            
            chunk_samples = 5 * 32000
            shift_samples = int(0.5 * 32000) 
            
            total_samples = waveform.shape[1]
            num_chunks = total_samples // chunk_samples 
            
            file_probs = []
            batch_size = 16 
            
            for b in range(0, num_chunks, batch_size):
                wave_chunks_base = []
                wave_chunks_left = []
                wave_chunks_right = []
                
                for i in range(b, min(b + batch_size, num_chunks)):
                    start = i * chunk_samples
                    end = start + chunk_samples
                    
                    wave_chunks_base.append(get_padded_chunk(waveform, start, end, chunk_samples))
                    wave_chunks_left.append(get_padded_chunk(waveform, start - shift_samples, end - shift_samples, chunk_samples))
                    wave_chunks_right.append(get_padded_chunk(waveform, start + shift_samples, end + shift_samples, chunk_samples))
                    
                batch_base = torch.stack(wave_chunks_base).to(device)
                batch_left = torch.stack(wave_chunks_left).to(device)
                batch_right = torch.stack(wave_chunks_right).to(device)
                
                with torch.no_grad():
                    probs_base = torch.sigmoid(model(batch_base)).cpu().numpy()
                    probs_left = torch.sigmoid(model(batch_left)).cpu().numpy()
                    probs_right = torch.sigmoid(model(batch_right)).cpu().numpy()
                    
                    probs_ensemble = (probs_base + probs_left + probs_right) / 3.0
                    file_probs.append(probs_ensemble)
            
            if len(file_probs) > 0:
                file_probs = np.vstack(file_probs)
                smoothed_probs = apply_temporal_smoothing(file_probs, sigma=1.0)
                
                for i in range(num_chunks):
                    end_second = (i + 1) * 5
                    row_id = f"{file_stem}_{end_second}"
                    predictions_dict[row_id] = smoothed_probs[i] 
                    
            del waveform
            gc.collect()
            
        except Exception as e:
            print(f"Error processing {file}: {e}")

for idx, row in sample_df.iterrows():
    r_id = row['row_id']
    if r_id in predictions_dict:
        pred_probs = predictions_dict[r_id]
        for col in expected_species_columns:
            if col in species_to_idx:
                sample_df.at[idx, col] = pred_probs[species_to_idx[col]]

sample_df.to_csv('submission.csv', index=False)
print("Submission.csv generated")